# External validation
Verify the frozen release, inspect Brazil and other markets, then reproduce all results offline.

In [ ]:
from pathlib import Path
import sys, subprocess, json, hashlib, zipfile, shutil
import polars as pl
root = Path.cwd()
if not (root / "RELEASE_MANIFEST.json").exists():
    matches = list(Path("/kaggle/input").rglob("RELEASE_MANIFEST.json")) if Path("/kaggle/input").exists() else []
    if matches: root = matches[0].parent
    elif (root.parent / "RELEASE_MANIFEST.json").exists(): root = root.parent
    else: raise FileNotFoundError("Select the extracted release root")
manifest = json.loads((root / "RELEASE_MANIFEST.json").read_text())
if any(not (root / row["path"]).is_file() for row in manifest["files"]):
    expanded = Path("/kaggle/working/expanded-release") if Path("/kaggle/working").exists() else Path("expanded-release")
    expanded.mkdir(exist_ok=False)
    for row in manifest["files"]:
        relative = Path(row["path"])
        target = (expanded / relative).resolve()
        if not target.is_relative_to(expanded.resolve()): raise ValueError("Invalid release path")
        direct = root / relative
        if direct.is_file(): content = direct.read_bytes()
        else:
            archive_path = root / (relative.parts[0] + ".zip")
            with zipfile.ZipFile(archive_path) as archive:
                candidates = [relative.as_posix(), Path(*relative.parts[1:]).as_posix()]
                names = [name for name in candidates if name in archive.namelist()]
                if len(names) != 1: raise ValueError("Missing or ambiguous archived member: " + str(relative))
                content = archive.read(names[0])
        if hashlib.sha256(content).hexdigest() != row["sha256"]: raise ValueError("Checksum mismatch")
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(content)
    (expanded / "RELEASE_MANIFEST.json").write_text(json.dumps(manifest))
    if (root / "SHA256SUMS.txt").is_file(): shutil.copyfile(root / "SHA256SUMS.txt", expanded / "SHA256SUMS.txt")
    root = expanded
subprocess.run([sys.executable, str(root / "code/verify_release.py"), str(root)], check=True)


In [ ]:
markets = pl.read_parquet(root / "data/results/country-metrics.parquet")
markets.filter(pl.col("market_code").is_in(["BR", "US", "GB", "DE", "JP"]))


In [ ]:
output = Path("/kaggle/working/reproduced") if Path("/kaggle/working").exists() else Path("reproduced")
subprocess.run([sys.executable, str(root / "code/reproduce.py"), "--release", str(root), "--output", str(output)], check=True)
